# Monarch Simulator: Complete Guide

This notebook covers everything you need to know about the Monarch Simulator —
how it works, how to use it, its benefits, and how to pitch it to partners.

---

## 1. What Is The Monarch Simulator?

The Monarch Simulator is a **discrete-event simulator** that lets you simulate distributed GPU
training workloads **without requiring actual GPU hardware**.

It acts as a **drop-in replacement** for the real `ProcessBackend`, intercepting all controller
commands (tensor operations, collectives, stream synchronization) and simulating their execution
on virtual workers with virtual GPU streams.

| Aspect | Details |
|--------|----------|
| **Location** | `python/monarch/simulator/` |
| **Public API** | `monarch.Simulator`, `monarch.set_meta` |
| **Language** | Pure Python (no Rust simulation code) |
| **Tensor backend** | PyTorch `FakeTensorMode` for shape propagation without materializing data |
| **Trace output** | Chrome Trace JSON — viewable in `chrome://tracing` or Perfetto |

---
## 2. Architecture Overview

### 2.1 How a Simulator Instance is Created

The `Simulator()` factory function (`python/monarch/simulator/interface.py`) creates:

1. A **`SimulatorController`** — replaces the real ProcessBackend
2. A **`Client`** — wrapping that controller
3. A **`DeviceMesh`** — identical API to a real device mesh

```
monarch.Simulator(hosts=2, gpus=4)
        │
        ├── SimulatorController (mock backend)
        │       └── Simulator (discrete event engine)
        │              ├── Worker 0
        │              │     ├── Stream 0  →  [Task Queue]
        │              │     └── Stream 1  →  [Task Queue]
        │              ├── Worker 1
        │              │     ├── Stream 0  →  [Task Queue]
        │              │     └── Stream 1  →  [Task Queue]
        │              └── ... (one worker per simulated GPU)
        ├── Client
        └── DeviceMesh  ← you use this, same API as real
```

### 2.2 Execution Model

From `python/monarch/simulator/README.md`:

- The **Simulator** maintains multiple **Worker** objects (one per simulated GPU)
- Each **Worker** has multiple **Stream** objects (like CUDA streams)
- Each **Stream** has a **task queue** with a state machine:

```
PENDING  →  READY  →  EXECUTING  →  EXECUTED
```

- A task becomes `READY` when all its dependencies are fulfilled
- **Computation tasks** finish immediately after executing
- **Collective ops** (all_reduce, etc.) wait for **all** participating workers
- A **trace event** is recorded after each task finishes

### 2.3 Memory Model

- Only **GPU memory** is tracked (CPU ops assumed zero-cost)
- Uses `WorkerStorageTracker` for unique storage tracking — avoids double-counting for tensor views
- Memory is allocated when a task executes, freed when `DeleteRefs` arrives

### 2.4 Communication Model

Defined in `python/monarch/simulator/communication_model.py`:

| Parameter | Default (H100) |
|-----------|----------------|
| NVLink bandwidth (intra-node) | 900 GB/s |
| InfiniBand bandwidth (inter-node) | 50 GB/s |
| Automatic intra/inter-node selection | Yes |

Supported collectives: `ring_allreduce`, `reduce_scatter`, `allgather`, `alltoall`, `p2p`

---
## 2.5 Deep Dive: IR Graph and DAGs Explained

This section explains three terms you'll encounter throughout the simulator:
**IR**, **DAG**, **Control DAG**, and **Data DAG**. If these are new to you, read this
section carefully — they are central to how the simulator records and analyzes your workload.

### What is an "IR Graph"?

**IR** stands for **Intermediate Representation**. It is a structured, inspectable snapshot
of *everything your training pipeline did* during a simulation run.

Think of it like a detailed flight recorder for your distributed training:
- Every operation that ran (matrix multiply, all-reduce, send tensor, etc.)
- On which worker and stream it ran
- What tensors it read and produced
- What it depended on before it could start
- How long it took (real or estimated)

The IR Graph is implemented in `python/monarch/simulator/ir.py` as the `IRGraph` class.
You enable it by passing `build_ir=True` to the `Simulator()` factory.

### What is a "DAG"?

**DAG** stands for **Directed Acyclic Graph**. It's a fundamental computer science concept:

- **Directed**: Edges have a direction (A -> B means "A comes before B" or "A feeds into B")
- **Acyclic**: No cycles — you can't follow edges and end up back where you started
- **Graph**: A collection of nodes connected by edges

In the simulator context, a DAG captures **ordering relationships** — which operations
must complete before others can begin, and which tensors flow from one operation to another.

```
Simple DAG example:

    [matmul]  ──>  [all_reduce]  ──>  [optimizer_step]
       │                                     ^
       └──────>  [loss_compute]  ────────────┘

This says: matmul must finish before all_reduce AND loss_compute can start.
optimizer_step can only start after BOTH all_reduce and loss_compute are done.
```

### The Two DAGs in the IR Graph

The Monarch Simulator's `IRGraph` maintains **two separate DAGs** that capture different
aspects of your workload:

#### 1. Control DAG (`IRGraph.control_dag`) — "What ran, in what order?"

The Control DAG tracks **command execution** across workers and streams. Each node is a
`Command` that records:

| Field | Meaning |
|-------|---------|
| `worker_rank` | Which simulated GPU ran this command |
| `stream_name` | Which CUDA stream on that GPU |
| `command_id` | Unique ID for this command |
| `command_name` | What kind of operation (e.g., `CallFunction: aten.mm`, `Reduce: all_reduce`, `SendTensor: 7`) |
| `devices` | All GPU device IDs involved (important for collectives) |
| `control_dependencies` | List of command IDs that **must finish before** this command can start |
| `duration` | Execution time in microseconds (set via profiling or `import_timing()`) |
| `traceback` | Python call stack at the point this command was issued |

**Example:** If your training does `y = mm(x, x)` then `z = all_reduce(y)`, the Control DAG has:
```
Command(id=0, name="CallFunction: aten.mm", dependencies=[])
    └──> Command(id=1, name="Reduce: all_reduce", dependencies=[0])
```
Command 1 depends on Command 0 — the all_reduce can't start until the matmul finishes.

#### 2. Data DAG (`IRGraph.data_dag`) — "Where did each tensor come from and go?"

The Data DAG tracks the **lifecycle of tensors and storage** through six event types:

| Event Type | What It Records |
|------------|----------------|
| `StorageCreationEvent` | A new GPU memory allocation was made |
| `StorageDeletionEvent` | GPU memory was freed |
| `TensorCreationEvent` | A new tensor was created (references some storage) |
| `TensorAccessEvent` | A tensor was read by an operation |
| `TensorMutationEvent` | A tensor was modified in-place |
| `TensorDeletionEvent` | A tensor reference was dropped |

Each event records the `command_id` (linking it to the Control DAG), `storage_id`,
`DTensorRef`, tensor `dims` and `dtype`, and which `devices` were involved.

**Why two DAGs?** Because they answer different questions:
- **Control DAG**: "Is my pipeline bottlenecked on communication? Are streams being utilized efficiently?"
- **Data DAG**: "Where is memory being allocated? When is it freed? Which ops share storage?"

### Visual: How the Two DAGs Relate

```
CONTROL DAG (execution order)          DATA DAG (tensor lifecycle)
================================       ================================

Cmd 0: CallFunction: aten.mm           StorageCreation(storage=0, cmd=0)
  │    worker=0, stream=default         TensorCreation(ref=1, storage=0, cmd=0)
  │                                     TensorAccess(ref=1, storage=0, cmd=0)
  │
  v
Cmd 1: Reduce: all_reduce              TensorAccess(ref=1, storage=0, cmd=1)
  │    worker=0, stream=default           (reads the tensor produced by cmd 0)
  │    devices=[0,1,2,3]
  │
  v
Cmd 2: CallFunction: aten.add_         TensorMutation(ref=1, storage=0, cmd=2)
  │    worker=0, stream=default           (in-place modification)
  │
  v
Cmd 3: DeleteRefs                       TensorDeletion(ref=1, storage=0, cmd=3)
       worker=0, stream=default          StorageDeletion(storage=0, cmd=3)
                                           (memory freed)
```

### Timing Keys: How Operations Are Identified

The IR Graph generates **timing keys** — unique identifiers for each type of operation
that include the operation name, tensor shapes, dtypes, and (for collectives) device count.

| Command Type | Timing Key Format | Example |
|-------------|-------------------|---------|
| `CallFunction` | `CallFunction:{func}:{input_shapes}:{dtype}` | `CallFunction:aten.mm:(3x4)x(4x5):float32` |
| `Reduce` | `Reduce:{type}:{shape}:{dtype}:{num_devices}` | `Reduce:reduce_scatter:(1024x1024):float32:8` |
| `SendTensor` | `SendTensor:{shape}:{dtype}` | `SendTensor:(512x512):float32` |
| `Borrow*` | `{borrow_type}` | `BorrowCreate` |

These keys are essential for the **offline profiling workflow**: export timing keys,
benchmark each one externally, then import the results back.

### The Offline Profiling Workflow

The IR Graph enables a powerful workflow where you separate *what runs* from *how long it takes*:

```
Step 1: Run simulation with build_ir=True
          ↓
Step 2: Export unique command types
        ir.export_command_types("commands.json")
          ↓
Step 3: Benchmark each timing key on real hardware
        (or use include_timing=True to auto-profile)
          ↓
Step 4: Import timing data back
        ir.import_timing("timing.json")
          ↓
Step 5: Export timing-accurate trace
        ir.export_dag_json_timed("timed_trace.json")
          ↓
Step 6: Visualize in chrome://tracing or Perfetto
```

### Export Formats

The IR Graph supports multiple export formats for different analysis needs:

| Method | Output | Use Case |
|--------|--------|----------|
| `export_dag_json(path)` | Chrome Trace JSON (fixed-width) | Control flow visualization |
| `export_dag_json_timed(path)` | Chrome Trace JSON (real timing) | Performance analysis |
| `export_command_types(path)` | JSON catalog of unique ops | Profiling preparation |
| `export_borrows_csv(path)` | TSV file | Cross-stream borrow analysis |
| `export_sendtensors_csv(path)` | TSV file | Tensor send pattern analysis |
| `export_data_csv(path)` | TSV file | Storage/tensor lifecycle analysis |
| `export_data_timeline_csv(path)` | TSV file | Chronological data event timeline |
| `export_memory_viz(path)` | Memory snapshot JSON | Memory visualization (compatible with `torch.cuda._memory_viz`) |
| `import_timing(path_or_dict)` | (input) | Apply external timing to all commands |

---
## 3. Key Modules Reference

| Module | File | Purpose |
|--------|------|----------|
| **Interface** | `simulator/interface.py` | `Simulator()` factory — public entry point |
| **Core Engine** | `simulator/simulator.py` | `Simulator`, `SimulatorController`, `SimulatorInterface`, mode enums |
| **Workers** | `simulator/worker.py` | `Worker`, `WorkerGroup`, `Stream` — multi-stream task execution |
| **Tasks** | `simulator/task.py` | `Task`, `EventTask`, `Borrow`, `WorkerTaskManager` — task lifecycle |
| **Tensors** | `simulator/tensor.py` | `DTensorRef`, `FakeTensorTracker`, `WorkerStorageTracker`, `TensorManager` |
| **Traces** | `simulator/trace.py` | `TraceEvent`, Chrome Trace JSON, `MemoryViewer`, Perfetto upload |
| **Profiling** | `simulator/profiling.py` | `RuntimeProfiler` (CUDA), `FakeRuntimeProfiler`, `RuntimeEstimator` |
| **Communication** | `simulator/communication_model.py` | Latency/bandwidth models for all collective types |
| **IR Graph** | `simulator/ir.py` | Control DAG + Data DAG, timing import/export, CSV exports |
| **Command History** | `simulator/command_history.py` | Record/replay controller commands |
| **Config** | `simulator/config.py` | `set_meta()` context manager for annotating tasks |
| **Mock Controller** | `simulator/mock_controller.py` | Fake controller for sending/receiving without real workers |
| **Actor Mocks** | `_src/actor/mock.py` | `patch_actor`, `patch_tensor_engine` — actor/engine replacement |

---
## 4. Operating Modes

### 4.1 Backend Modes (`SimulatorBackendMode`)

| Mode | What It Does |
|------|-------------|
| `SIMULATE` *(default)* | Simulate execution + dump trace files |
| `SIMULATE_WITH_REPORT_ONLY` | Simulate but produce no trace output |
| `COMMAND_HISTORY` | Only record commands for later replay |
| `EVERYTHING` | Simulate + record commands |

### 4.2 Trace Modes (`SimulatorTraceMode`)

| Mode | What It Traces |
|------|---------------|
| `STREAM_ONLY` *(default)* | GPU stream activity per worker |
| `CONTROLLER_TRACE_ONLY` | Controller timeline only |
| `EVERYTHING` | Both streams and controller |

---
## 5. Two Simulation Approaches

Monarch gives you **two complementary tools** depending on what level of fidelity you need.

### Approach A: `patch_actor` — Mock Actor Logic

Replace real actor classes with lightweight mocks. Best for testing **actor-level logic**
(message passing, coordination, scheduling) without real computation.

**When to use:** You want to test that actors communicate correctly, that your GRPO pipeline
orchestrates generators and learners properly, etc.

In [ ]:
# Approach A: patch_actor example
# Replace real actors with mock versions that skip GPU work

import asyncio
from monarch._src.actor.mock import patch_actor
from monarch.actor import Actor, endpoint


# --- Real actors (would normally need GPUs) ---
class RealLearner(Actor):
    @endpoint
    async def step(self):
        # Heavy GPU training step
        pass


# --- Mock actors (no GPUs needed) ---
class MockLearner(Actor):
    @endpoint
    async def step(self):
        import torch
        return torch.tensor(1.0)  # Fake result


# --- Usage as decorator ---
@patch_actor(RealLearner, MockLearner)
async def simulate_with_decorator():
    # Any spawn(RealLearner) will spawn MockLearner instead
    pass  # your training pipeline here


# --- Usage as context manager ---
async def simulate_with_context_manager():
    with patch_actor(RealLearner, MockLearner):
        pass  # your training pipeline here


# --- Multiple patches can be stacked ---
# @patch_actor(Learner, MockLearner)
# @patch_actor(Generator, MockGenerator)
# async def simulate_all():
#     await grpo_main()

### Approach B: `patch_tensor_engine` — Simulated Tensor Engine

Replace the real tensor engine with the Simulator. The actor logic runs as-is but
**all tensor operations are simulated**. Best for testing **full training pipelines**
including tensor operations, memory, and communication.

**When to use:** You want to profile execution time, memory usage, and collective
communication of your full pipeline without GPUs.

In [ ]:
# Approach B: patch_tensor_engine example
# The actor logic runs unchanged, but the tensor engine is simulated

import asyncio
from monarch._src.actor.mock import patch_tensor_engine


# --- Usage as decorator (auto-creates Simulator matching proc_mesh dims) ---
# @patch_tensor_engine(GRPOTrainer)
# async def simulate():
#     await grpo_main()  # Full pipeline, zero GPUs!


# --- Usage as context manager ---
# with patch_tensor_engine(GRPOTrainer):
#     await grpo_main()


# --- With a custom tensor engine factory ---
def my_custom_engine(proc_mesh):
    from monarch.simulator.interface import Simulator
    sim = Simulator(
        hosts=2,
        gpus=8,
        trace_path="custom_trace.json",
        build_ir=True,
    )
    return sim.mesh

# with patch_tensor_engine(MyActor, my_custom_engine):
#     await train()

---
## 6. Hands-On Examples

### Example 1: Basic Simulator — Create a Simulated DeviceMesh

In [ ]:
import monarch
import torch

# Create a simulator with 2 hosts, 4 GPUs each
sim = monarch.Simulator(
    hosts=2,
    gpus=4,
    trace_path="basic_trace.json",   # Output trace file
    trace_mode="STREAM_ONLY",        # Trace GPU streams
)

# sim.mesh is a DeviceMesh — same API as the real thing
mesh = sim.mesh

# Use it like a real mesh
with mesh.activate(), torch.device("cuda"):
    x = torch.randn(100, 100)
    y = torch.mm(x, x)

# Shut down — this writes the trace file
mesh.exit()

# Open basic_trace.json in chrome://tracing or Perfetto to see the timeline

### Example 2: Annotate Phases with `set_meta`

In [ ]:
import monarch
import torch

sim = monarch.Simulator(hosts=1, gpus=2, trace_path="annotated_trace.json")
mesh = sim.mesh

with mesh.activate(), torch.device("cuda"):

    # Tag tasks with meaningful phase names — they appear in the trace
    with monarch.set_meta("forward_pass"):
        x = torch.randn(256, 256)
        y = torch.mm(x, x)

    with monarch.set_meta("backward_pass"):
        grad = torch.randn(256, 256)
        dx = torch.mm(grad, x.T)

    with monarch.set_meta("optimizer_step"):
        x = x - 0.01 * dx

mesh.exit()

### Example 3: IR Graph for Offline Analysis

In [ ]:
import monarch
import torch

# Enable IR graph construction
sim = monarch.Simulator(hosts=2, gpus=4, build_ir=True)
mesh = sim.mesh

with mesh.activate(), torch.device("cuda"):
    x = torch.randn(100, 100)
    y = torch.mm(x, x)

mesh.exit()

# The IR graph has a dual-DAG structure:
#   - Control DAG: command execution order
#   - Data DAG: tensor/storage lifecycle
#
# Export for analysis:
# sim.export_ir("ir_graph.pt")
#
# Export command types with timing keys:
# ir.export_command_types("commands.json", include_timing=True)
#
# Import custom timing from benchmarks:
# ir.import_timing("custom_timing.json")
#
# Export timed trace:
# ir.export_dag_json_timed("timed_trace.json")

### Example 3b: IR Graph — Inspecting the Control DAG and Data DAG

This example shows how to directly inspect the two DAGs after a simulation run.
See **Section 2.5** above for a full explanation of what IR, DAG, Control DAG, and Data DAG mean.

In [ ]:
# Example 3b: Working with the IR Graph's two DAGs
#
# After running a simulation with build_ir=True, you can inspect both DAGs directly.

from monarch.simulator.ir import IRGraph, Command

# --- Creating an IRGraph (normally done automatically by the Simulator) ---
ir = IRGraph()

# --- The Control DAG: list of Command nodes ---
# Each Command records: worker_rank, stream_name, command_id, command_name,
#                        devices, control_dependencies, traceback, duration
#
# After simulation, inspect it:
# for cmd in ir.control_dag:
#     print(f"  Cmd {cmd.command_id}: {cmd.command_name}")
#     print(f"    Worker: {cmd.worker_rank}, Stream: {cmd.stream_name}")
#     print(f"    Depends on: {cmd.control_dependencies}")
#     print(f"    Devices: {cmd.devices}")
#     print(f"    Duration: {cmd.duration} us")

# --- The Data DAG: list of tensor/storage lifecycle events ---
# Event types: StorageCreation, StorageDeletion, TensorCreation,
#              TensorAccess, TensorMutation, TensorDeletion
#
# for event in ir.data_dag:
#     print(f"  {type(event).__name__}: storage={event.storage_id}, cmd={event.command_id}")

# --- Full offline profiling workflow ---
# Step 1: Run simulation with build_ir=True (see Example 3 above)
#
# Step 2: Export unique command types for benchmarking
# ir.export_command_types("commands.json")
#   -> Produces: {"command_types": [{"timing_key": "CallFunction:aten.mm:(100x100)x(100x100):float32", "count": 8, ...}]}
#
# Step 3: Import timing data (from benchmarks or manual estimates)
# ir.import_timing({
#     "CallFunction:aten.mm:(100x100)x(100x100):float32": 1500,  # 1500 us
#     "Reduce:all_reduce:(100x100):float32:8": 5000,              # 5000 us
#     "BorrowCreate": 10,                                          # 10 us
# })
#
# Step 4: Export timing-accurate Chrome Trace
# ir.export_dag_json_timed("timed_trace.json")
#   -> Open in chrome://tracing or Perfetto for visual analysis
#
# Step 5: Or export fixed-width trace (for control flow visualization only)
# ir.export_dag_json("control_flow_trace.json")
#
# Step 6: Export CSV reports for detailed analysis
# ir.export_borrows_csv("borrows.tsv")         # Cross-stream borrow patterns
# ir.export_sendtensors_csv("sends.tsv")       # Tensor send patterns
# ir.export_data_csv("tensor_lifecycle.tsv")    # Full tensor/storage lifecycle
# ir.export_data_timeline_csv("timeline.tsv")   # Chronological event timeline
# ir.export_memory_viz("memory_snapshot.json")  # Memory visualization
#
# Step 7: Auto-profile on GPU (if CUDA available) and apply timing in one step
# ir.export_command_types("commands_with_timing.json", include_timing=True)
#   -> Profiles compute kernels on GPU, estimates collectives, auto-applies timing

print("IR Graph key concepts:")
print("  IR    = Intermediate Representation (structured recording of all operations)")
print("  DAG   = Directed Acyclic Graph (ordered dependency graph, no cycles)")
print("  Control DAG = What ran, in what order, on which worker/stream")
print("  Data DAG    = Where each tensor was created, read, mutated, and freed")

### Example 4: GRPO Training Simulation (from repo examples)

The repo includes real-world examples in `docs/source/examples/`:

In [ ]:
# Example from docs/source/examples/grpo_sim.py
# Simulates GRPO training without real GPUs using patch_actor

import asyncio
from typing import Any, Dict, Optional

import torch
from monarch._src.actor.mock import patch_actor
from monarch.actor import Actor, endpoint
from monarch.rdma import RDMABuffer

# In real code, you'd import from your actual GRPO module:
# from my_training.grpo_actor import Generator, Learner, ReplayBuffer, main as grpo_main


# Mock the Learner — returns fake weights and loss
class MockLearner(Actor):
    def __init__(self, replay_buffer):
        self.replay_buffer = replay_buffer
        self.generators = None

    @endpoint
    async def weights_handle(self) -> Dict[str, RDMABuffer]:
        return {}

    @endpoint
    async def step(self) -> torch.Tensor:
        return torch.tensor(1.0)  # Fake loss

    @endpoint
    async def init_generators(self, generators) -> None:
        self.generators = generators


# Mock the Generator — skips actual generation
class MockGenerator(Actor):  # would inherit from Generator
    @endpoint
    async def generate(self, state: torch.Tensor) -> None:
        pass


# Stack decorators to mock multiple actors at once
# @patch_actor(Learner, MockLearner)
# @patch_actor(Generator, MockGenerator)
# async def simulate():
#     await grpo_main()  # Runs the full GRPO pipeline, zero GPUs!
#
# asyncio.run(simulate())

### Example 5: GRPO with Simulated Tensor Engine (from repo examples)

In [ ]:
# Example from docs/source/examples/grpo_te_sim.py
# Simulates GRPO with a full simulated tensor engine

import asyncio
from monarch._src.actor.mock import patch_tensor_engine

# In real code:
# from my_training.grpo_actor_te import GRPOTrainer, main as grpo_te_main


# Decorator syntax — creates a Simulator matching proc_mesh dimensions
# @patch_tensor_engine(GRPOTrainer)
# async def simulate_with_tensor_engine():
#     await grpo_te_main()


# Context manager syntax
# async def main():
#     with patch_tensor_engine(GRPOTrainer):
#         await grpo_te_main()
#
# asyncio.run(main())

print("See docs/source/examples/grpo_te_sim.py for the full runnable example")
print("Run with: buck2 run //monarch/docs/source/examples:grpo_te_sim")

### Example 6: Custom Communication Config

In [ ]:
# You can customize the network model for different hardware

from monarch.simulator.communication_model import (
    NetworkConfig,
    estimate_collective_time_us,
    estimate_send_time_us,
)

# Default H100 config
h100_config = NetworkConfig(
    intra_node_bw_GBps=900,   # NVLink bandwidth
    inter_node_bw_GBps=50,    # InfiniBand bandwidth
    latency_us=5,             # Per-hop latency
    gpus_per_node=8,          # GPUs per node
    contention_factor=1.0,    # Network contention multiplier
)

# Estimate time for a 1GB all-reduce across 16 GPUs on 2 nodes
# time_us = estimate_collective_time_us(
#     collective_type="all_reduce",
#     data_size_bytes=1e9,
#     num_ranks=16,
#     config=h100_config,
# )
# print(f"Estimated all-reduce time: {time_us:.0f} us")

### Example 7: Profiling with Custom Timing

In [ ]:
from monarch.simulator.profiling import RuntimeEstimator

# The RuntimeEstimator provides configurable timing for simulated ops
# Default timings:
#   - CallFunction:  10 us
#   - Reduce:       100 us
#   - SendTensor:   100 us

# You can set custom timing per operation type:
# estimator = RuntimeEstimator()
# estimator.set_custom_timing("CallFunction", 50)   # 50 us for compute ops
# estimator.set_custom_timing("Reduce", 200)        # 200 us for reductions

# For real GPU profiling (when CUDA is available):
# from monarch.simulator.profiling import RuntimeProfiler
# profiler = RuntimeProfiler()  # Spawns CUDA workers to benchmark kernels

# For no-GPU fallback:
# from monarch.simulator.profiling import FakeRuntimeProfiler
# profiler = FakeRuntimeProfiler()  # Uses FakeTensorMode

print("Profiling options:")
print("  RuntimeEstimator  — configurable fixed timing (no GPU needed)")
print("  RuntimeProfiler   — real CUDA benchmarking (GPU required)")
print("  FakeRuntimeProfiler — FakeTensorMode fallback (no GPU needed)")

---
## 7. Benefits Summary

| # | Benefit | Description |
|---|---------|-------------|
| 1 | **No GPU required** | Develop and test training pipelines on a laptop or CPU-only machine |
| 2 | **Fast iteration** | Skip GPU allocation queues; get instant feedback |
| 3 | **Execution traces** | Visualize multi-GPU, multi-stream execution timelines (Chrome Trace / Perfetto) |
| 4 | **Memory profiling** | Catch OOM issues before running on real hardware |
| 5 | **Communication modeling** | Realistic NVLink/IB latency and bandwidth estimates |
| 6 | **Drop-in replacement** | Same `DeviceMesh` API; switch between real and simulated with one line |
| 7 | **Cross-process propagation** | Mocks automatically propagate to subprocess workers |
| 8 | **Nestable & composable** | Stack multiple `patch_actor` / `patch_tensor_engine` decorators |
| 9 | **CI/CD friendly** | Runs without GPUs — add distributed training tests to every PR |
| 10 | **IR graph analysis** | Dual-DAG (control + data) for offline analysis, timing import/export |

---
## 8. Quick Reference: API Cheat Sheet

```python
import monarch
from monarch._src.actor.mock import patch_actor, patch_tensor_engine

# --- Create a simulator ---
sim = monarch.Simulator(
    hosts=2,                          # Number of hosts
    gpus=4,                           # GPUs per host
    simulate_mode="SIMULATE",         # SIMULATE | SIMULATE_WITH_REPORT_ONLY | COMMAND_HISTORY | EVERYTHING
    trace_mode="STREAM_ONLY",         # STREAM_ONLY | CONTROLLER_TRACE_ONLY | EVERYTHING
    trace_path="trace.json",          # Output trace file path
    upload_trace=False,               # Upload to Perfetto
    group_workers=False,              # Deduplicate identical workers
    build_ir=False,                   # Build IR graph for analysis
)
mesh = sim.mesh                       # DeviceMesh — use like the real thing

# --- Annotate phases ---
with monarch.set_meta("phase_name"):
    ...  # tasks created here are tagged in the trace

# --- Mock actors ---
with patch_actor(RealActor, MockActor):
    ...  # spawn(RealActor) → spawns MockActor instead

# --- Simulate tensor engine ---
with patch_tensor_engine(MyActor):
    ...  # tensor engine auto-replaced with Simulator

# --- Shut down ---
mesh.exit()                           # Writes trace file and cleans up
```

---
## 9. Partner Speech Notes & Pitch Material

Use the sections below when presenting the Monarch Simulator to partners, stakeholders,
or potential adopters.

### 9.1 Elevator Pitch (30 seconds)

> "Monarch Simulator lets you develop, test, and profile distributed GPU training
> pipelines **without any GPUs**. One line of code swaps your real backend for a
> simulated one. You get execution traces, memory profiling, and communication
> modeling — all on your laptop. It means faster iteration, earlier bug detection,
> and no waiting for GPU cluster allocation."

### 9.2 Key Talking Points

#### Point 1: Shift-Left GPU Development
> "Today, you can't test distributed training code until you get a GPU allocation.
> With Monarch Simulator, you can validate your entire pipeline — actor coordination,
> tensor operations, collective communication — on **day one, on your laptop**."

#### Point 2: Two Levels of Simulation
> "We give you two tools:
> - `patch_actor` for **lightweight mocking** of actor logic
> - `patch_tensor_engine` for **full tensor engine simulation**
>
> Choose the level of fidelity you need."

#### Point 3: Production-Quality Traces
> "The simulator outputs Chrome Trace JSON — the same format used by CUDA profiling
> tools. You can visualize multi-worker, multi-stream execution timelines in
> **Perfetto** or `chrome://tracing` **before you ever touch a GPU**."

#### Point 4: Realistic Communication Modeling
> "We model NVLink and InfiniBand bandwidth, intra-node vs inter-node topology,
> and all collective operation types (all-reduce, reduce-scatter, all-gather,
> all-to-all). Default configs are tuned for **H100 clusters**."

#### Point 5: Memory Safety Net
> "The simulator tracks GPU memory allocation with storage deduplication — catching
> potential **OOMs before they happen** on expensive hardware."

#### Point 6: Zero Code Changes
> "It's a **drop-in replacement**. Your training code uses the same `DeviceMesh` API.
> Wrap your entry point in `patch_tensor_engine(MyActor)` and you're simulating.
> Remove it and you're on real GPUs."

#### Point 7: CI/CD Integration
> "Because it runs without GPUs, you can add **simulation-based tests to your CI
> pipeline**. Catch distributed training regressions on every PR, not just in
> nightly GPU runs."

### 9.3 Differentiators vs. Alternatives

| Monarch Simulator | Alternative | Why Monarch Wins |
|-------------------|-------------|------------------|
| Full discrete-event simulation | PyTorch `FakeTensorMode` alone | Simulates **multi-worker distributed execution** with streams, collectives, communication |
| Execution traces + memory profiles | Simple mocking frameworks | Provides **visual profiling artifacts**, not just pass/fail |
| Works without hardware | GPU profilers (nsight, etc.) | Iterate at **CPU speeds**, no hardware booking needed |
| Drop-in `DeviceMesh` API | Custom simulation harnesses | **Zero code changes** to switch between real and simulated |

### 9.4 Target Audiences

| Audience | Key Message |
|----------|-------------|
| Teams developing new distributed training pipelines | "Validate architecture and orchestration without waiting for GPU allocation" |
| Teams debugging multi-GPU coordination issues | "Reproduce and inspect distributed bugs with full execution traces" |
| Teams wanting CI coverage for training code | "Add distributed training tests to every PR — no GPU CI runners needed" |
| Teams doing capacity planning | "Model different host/GPU configurations to find optimal topology" |
| Anyone blocked waiting for GPU allocation | "Start building today, on your laptop" |

### 9.5 Demo Script (5-minute live demo)

1. **Show the problem** (30s): "Here's a training pipeline that needs 8 GPUs to run."
2. **Add one line** (30s): Wrap with `@patch_tensor_engine(MyActor)` — "Now it runs on my laptop."
3. **Run it** (60s): Execute the simulated pipeline, show it completes without GPUs.
4. **Show the trace** (120s): Open the generated `trace.json` in Perfetto. Walk through:
   - Multi-worker stream timelines
   - Collective op synchronization points
   - Memory allocation/deallocation over time
5. **Key takeaway** (60s): "We just profiled a distributed training pipeline without a single GPU. This works in CI, on laptops, and anywhere you have Python."

---
## 10. File Reference: Where to Find Everything

### Core Simulator
```
python/monarch/simulator/
├── __init__.py              # Package init
├── interface.py             # Simulator() factory — PUBLIC ENTRY POINT
├── simulator.py             # Core engine, SimulatorController, mode enums
├── worker.py                # Worker, WorkerGroup, Stream
├── task.py                  # Task, EventTask, Borrow, WorkerTaskManager
├── tensor.py                # DTensorRef, FakeTensorTracker, memory tracking
├── trace.py                 # TraceEvent, Chrome Trace, MemoryViewer
├── profiling.py             # RuntimeProfiler, FakeRuntimeProfiler, RuntimeEstimator
├── communication_model.py   # Network latency/bandwidth models
├── ir.py                    # IRGraph — control DAG + data DAG
├── command_history.py       # Record/replay commands
├── mock_controller.py       # MockController base
├── config.py                # set_meta() context manager
├── utils.py                 # Utilities
└── README.md                # Design documentation
```

### Actor Mocking
```
python/monarch/_src/actor/mock.py   # patch_actor, patch_tensor_engine
```

### Examples
```
docs/source/examples/
├── grpo_sim.py              # GRPO simulation with patch_actor
├── grpo_te_sim.py           # GRPO simulation with patch_tensor_engine
└── grpo_forge_sim.py        # TorchForge GRPO with patch_actor
```

### Tests
```
python/tests/simulator/
├── test_simulator.py             # End-to-end simulation tests
├── test_worker.py                # Worker/Stream tests
├── test_task.py                  # Task lifecycle tests
├── test_profiling.py             # Profiling tests
├── test_ir.py                    # IR graph tests
├── test_communication_model.py   # Communication model tests
├── test_actor_mock.py            # patch_actor tests
└── test_tensor_engine_mock.py    # patch_tensor_engine tests
```

---

*Generated from the Monarch repository at `monarch_dev/`. Refer to the source files above for the most up-to-date implementation details.*